# PLANTEUR — demo

Load a patch and visualise the Sentinel-2 series with its semantic annotation, then
read the georeferenced labels and the Landsat arrays.

Set `PLANTEUR_ROOT` to the dataset folder.


In [ ]:
import sys, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
sys.path.append("code")
from dataloader import PLANTEUR_Dataset

PLANTEUR_ROOT = "/path/to/PLANTEUR"  # <- set me

VOID = 255
label_names = json.load(open("documentation/label_names.json"))

# colormap.txt is "<class id> <R> <G> <B>" with 0-255 channels, one line per trained class.
rgb = {}
for line in open("documentation/colormap.txt"):
    parts = line.split()
    if len(parts) == 4:
        rgb[int(parts[0])] = tuple(int(v) / 255 for v in parts[1:])

class_ids = sorted(rgb)                      # the 8 trained classes, 0-7
class_labels = [label_names[str(i)] for i in class_ids]
cmap = ListedColormap([rgb[i] for i in class_ids])
cmap.set_bad("white")                        # void is masked, not a class
print(len(class_ids), "trained classes:", class_labels)
print("void is", VOID, "->", label_names[str(VOID)], "(excluded from loss and metrics)")


In [ ]:
# Load the test split (S2). Use sats=["S2","S1"] for optical+radar, target="instance" for panoptic.
ds = PLANTEUR_Dataset(folder=PLANTEUR_ROOT, sats=["S2"], target="semantic", folds=[5])
ids = [int(i) for i in ds.id_patches]
print("patches in fold 5 (test):", len(ds))

# Most patches are mostly Background, so pick one that shows several crops and some void.
# Reading the label .npy files is cheap; the imagery is not.
def richness(i):
    u = set(np.unique(np.load(f"{PLANTEUR_ROOT}/ANNOTATIONS/TARGET_{i}.npy")).tolist())
    return len(u - {VOID}), VOID in u

pid = max(ids[:300], key=richness)
idx = ids.index(pid)
(data, dates), target = ds[idx]
s2 = data["S2"] if isinstance(data, dict) else data
labels = np.asarray(target)
print("using patch      :", pid)
print("S2 time series   :", tuple(s2.shape), "| target:", labels.shape, labels.dtype)
print("classes present  :", [label_names[str(v)] for v in np.unique(labels)])


In [ ]:
# S2 true colour (B04,B03,B02 = channels 2,1,0) beside the semantic labels.
# Pick the least-bright frame, which tends to be the least cloudy.
def stretch(chw):
    """Per-channel 2-98% stretch to [0,1]; per channel, so colours stay balanced."""
    out = np.zeros(chw.shape, np.float32)
    for c in range(chw.shape[0]):
        lo, hi = np.percentile(chw[c], [2, 98])
        out[c] = np.clip((chw[c] - lo) / (hi - lo + 1e-6), 0, 1)
    return out.transpose(1, 2, 0)

t = int(np.asarray(s2[:, 0].reshape(s2.shape[0], -1).mean(axis=1)).argmin())
img = stretch(np.asarray(s2[t][[2, 1, 0]]).astype(np.float32))

fig, ax = plt.subplots(1, 2, figsize=(9.5, 4.5))
ax[0].imshow(img); ax[0].set_title(f"Sentinel-2 patch {pid} (t={t})", fontsize=10); ax[0].axis("off")
im = ax[1].imshow(np.ma.masked_where(labels == VOID, labels), cmap=cmap,
                  vmin=-0.5, vmax=len(class_ids) - 0.5)
ax[1].set_title("Semantic labels (void = white)", fontsize=10); ax[1].axis("off")
cbar = fig.colorbar(im, ax=ax[1], ticks=class_ids); cbar.ax.set_yticklabels(class_labels)
plt.tight_layout(); plt.show()


## Georeferenced labels

`ANNOTATIONS_GEOTIFF/TARGET_<id>.tif` carries the same label values as the `.npy` plus
each patch's CRS and affine transform, with `nodata=255`. The imagery arrays share the
grid, so a patch's label GeoTIFF georeferences its `DATA_*` arrays too.

Patches span five UTM zones, so there is no single CRS for the dataset.


In [ ]:
import rasterio
from rasterio.warp import transform_bounds

with rasterio.open(f"{PLANTEUR_ROOT}/ANNOTATIONS_GEOTIFF/TARGET_{pid}.tif") as src_tif:
    tif = src_tif.read(1)
    masked = src_tif.read(1, masked=True)   # nodata=255 -> void comes back masked
    crs = src_tif.crs
    print("patch      :", pid)
    print("crs        :", crs)
    print("transform  :", tuple(round(v, 2) for v in src_tif.transform[:6]))
    print("bounds     :", tuple(round(v, 1) for v in src_tif.bounds))
    print("lon/lat    :", tuple(round(v, 5) for v in transform_bounds(crs, "EPSG:4326", *src_tif.bounds)))
    print("void pixels:", int(masked.mask.sum()))

print("identical to ANNOTATIONS/TARGET_%d.npy:" % pid, np.array_equal(tif, labels))
print("crs distribution:", json.load(open(f"{PLANTEUR_ROOT}/ANNOTATIONS_GEOTIFF/_crs_summary.json"))["crs_counts"])

# Labels straight from the GeoTIFF, with void masked out.
fig, ax = plt.subplots(figsize=(6.4, 4.5))
im = ax.imshow(np.ma.masked_where(tif == VOID, tif), cmap=cmap,
               vmin=-0.5, vmax=len(class_ids) - 0.5)
ax.set_title(f"TARGET_{pid}.tif  —  {crs}", fontsize=10); ax.axis("off")
cbar = fig.colorbar(im, ax=ax, ticks=class_ids); cbar.ax.set_yticklabels(class_labels)
plt.tight_layout(); plt.show()


## Landsat

Landsat-8 is published as two arrays because its bands have two resolutions: the
multispectral bands on a 20 m grid (64x64) and the panchromatic band on a 10 m grid
(128x128), both covering the same 1,280 m patch footprint. Both share one date list
per patch (`dates-LS`), and the Landsat count is independent of S2/S1 — patches in a
WRS-2 sidelap see about twice as many passes.


In [ ]:
ls  = np.load(f"{PLANTEUR_ROOT}/DATA_LS/LS_{pid}.npy", mmap_mode="r")
pan = np.load(f"{PLANTEUR_ROOT}/DATA_LS_PAN/LSPAN_{pid}.npy", mmap_mode="r")
print("DATA_LS     :", ls.shape, ls.dtype, "  bands B1,B2,B3,B4,B5,B6,B7,B9,B10,B11")
print("DATA_LS_PAN :", pan.shape, pan.dtype, "  band B8")

# Landsat true colour is B4,B3,B2 = channels 3,2,1 of DATA_LS.
k = int(np.asarray(ls[:, 1].reshape(ls.shape[0], -1).mean(axis=1)).argmin())
lsi  = stretch(np.asarray(ls[k][[3, 2, 1]]).astype(np.float32))
pani = stretch(np.asarray(pan[k]).astype(np.float32))[:, :, 0]

fig, ax = plt.subplots(1, 2, figsize=(9.5, 4.5))
ax[0].imshow(lsi); ax[0].set_title(f"Landsat multispectral, 20 m grid (t={k})", fontsize=10); ax[0].axis("off")
ax[1].imshow(pani, cmap="gray"); ax[1].set_title(f"Landsat panchromatic, 10 m grid (t={k})", fontsize=10); ax[1].axis("off")
plt.tight_layout(); plt.show()
